# Review Demand, Weather and Renewable-energy Inputs

**About this notebook:** this reviews the inputs to a selected PyPSA-Earth run. It is separate from the reviewed fixed-capacity interruption model.

**Purpose:** show demand over time, electricity production, wind and solar availability, the maximum renewable capacity allowed by land and sea assumptions, and the OpenStreetMap transmission data used to create model locations.

**Before running:** use the repository `.venv` kernel and provide the result plus the matching files under `pypsa-earth/resources/<run>`. The result and input files must come from the same run.

**User instructions:** set `RUN_NAME` or the file paths, copy missing files from ARC, inspect the tables and charts, and record the demand source, weather year, map resolution and land/sea assumptions when sharing results.

**Common adjustments:** you can choose another run and change how charts are grouped or displayed. Do not mix a weather or demand file from another run unless the difference is clearly stated.

The time-series charts show **MWh in each model time step**. For a three-hour run, each value is average MW multiplied by three hours.

## Copy missing input files from ARC

If the required files are not present locally, use the current ARC example below:

In [ ]:
from pathlib import Path
import copy
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr
import yaml
from shapely import wkt

warnings.filterwarnings("ignore", category=FutureWarning)


def _repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").exists() and (candidate / "pypsa-earth").exists():
            return candidate
    return Path("..").resolve()


def _deep_update(base, override):
    result = copy.deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = _deep_update(result[key], value)
        else:
            result[key] = value
    return result


REPO_ROOT = _repo_root()
RUN_NAME = "mauritius-year-1"
PYPSA_DIR = REPO_ROOT / "pypsa-earth"
RESULT_NETWORK = PYPSA_DIR / "results" / RUN_NAME / "networks" / "elec_s_12flex_ec_lcopt_3h.nc"
LOCAL_NETWORK = PYPSA_DIR / "networks" / RUN_NAME / "elec_s_12flex_ec_lcopt_3h.nc"
BASE_NETWORK_PATH = PYPSA_DIR / "networks" / RUN_NAME / "base.nc"
CLUSTERED_NETWORK_PATH = PYPSA_DIR / "networks" / RUN_NAME / "elec_s_12flex.nc"
NETWORK_PATH = RESULT_NETWORK if RESULT_NETWORK.exists() else LOCAL_NETWORK
RESOURCES = PYPSA_DIR / "resources" / RUN_NAME
PROFILE_DIR = RESOURCES / "renewable_profiles"
BASE_NETWORK_DIR = RESOURCES / "base_network"
BUS_REGION_DIR = RESOURCES / "bus_regions"
SHAPES_DIR = RESOURCES / "shapes"
DEMAND_PATH = RESOURCES / "demand_profiles.csv"
SCENARIO_CONFIG = PYPSA_DIR / "configs" / "scenarios" / "config.mauritius-year-1.yaml"
DEFAULT_CONFIG = PYPSA_DIR / "config.default.yaml"
DEMAND_LOG = PYPSA_DIR / "logs" / RUN_NAME / "build_demand_profiles.log"

if not NETWORK_PATH.exists():
    raise FileNotFoundError(f"Solved network not found: {NETWORK_PATH}")

with DEFAULT_CONFIG.open() as fh:
    default_config = yaml.safe_load(fh)
with SCENARIO_CONFIG.open() as fh:
    scenario_config = yaml.safe_load(fh)
config = _deep_update(default_config, scenario_config)

n = pypsa.Network(str(NETWORK_PATH))

TECH_COLORS = {
    "solar": "#f9d71c",
    "onwind": "#235ebc",
    "offwind-ac": "#6895dd",
    "offwind-dc": "#74c6f2",
    "ror": "#78d4cf",
    "hydro": "#08ad97",
    "OCGT": "#e05b5b",
    "CCGT": "#d35050",
    "coal": "#545454",
    "lignite": "#826a4a",
    "nuclear": "#ff8c00",
    "oil": "#2a2a2a",
    "biomass": "#baa741",
    "load shedding": "#ff0000",
}


def tech_color(carrier):
    return TECH_COLORS.get(carrier, "#888888")


def load_profile_dataset(path):
    # Load eagerly and close the NetCDF handle; repeated lazy opens can trigger HDF errors.
    with xr.open_dataset(path) as ds:
        return ds.load()


def styled_table(df, fmt=None, **kwargs):
    try:
        styler = df.style
        if fmt is not None:
            styler = styler.format(fmt, **kwargs)
        return styler
    except Exception:
        return df


snapshot_weights = n.snapshot_weightings.generators.reindex(n.snapshots).fillna(
    n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)
)
period_hours = sorted(snapshot_weights.dropna().unique().tolist())
period_label = f"{period_hours[0]:g} h" if len(period_hours) == 1 else "variable duration"

summary = pd.Series({
    "network_file": str(NETWORK_PATH.relative_to(REPO_ROOT)),
    "snapshots": len(n.snapshots),
    "snapshot_start": str(n.snapshots[0]),
    "snapshot_end": str(n.snapshots[-1]),
    "model_period": period_label,
    "buses": len(n.buses),
    "AC_buses": int(n.buses.carrier.eq("AC").sum()) if "carrier" in n.buses else len(n.buses),
    "loads": len(n.loads),
    "generators": len(n.generators),
})
display(summary.to_frame("value"))


## A readable view of demand and electricity production

For an annual chart:

- show demand at each model location as negative lines, so the geographic split is visible;
- show electricity production by technology as positive lines, because showing every technology at every location would be too crowded.

The second chart is a location check: total generation and total demand use the same colour for each model location.

In [ ]:
# Load as negative MWh per model period; generation as positive MWh per model period.
load_mwh = n.loads_t.p_set.mul(snapshot_weights, axis=0)
load_by_bus = load_mwh.T.groupby(n.loads.bus).sum().T
load_by_bus = load_by_bus.loc[:, load_by_bus.sum().sort_values(ascending=False).index]

# Positive dispatch only. Load shedding is shown separately if it is ever used.
gen_dispatch_mwh = n.generators_t.p.clip(lower=0.0).mul(snapshot_weights, axis=0)
gen_by_carrier = gen_dispatch_mwh.T.groupby(n.generators.carrier).sum().T
gen_by_carrier = gen_by_carrier.loc[:, gen_by_carrier.sum().sort_values(ascending=False).index]
gen_by_carrier = gen_by_carrier.loc[:, gen_by_carrier.sum() > 1e-6]

node_palette = plt.cm.tab10(np.linspace(0, 1, max(len(load_by_bus.columns), 1)))
node_colors = {bus: node_palette[i % len(node_palette)] for i, bus in enumerate(load_by_bus.columns)}

fig, ax = plt.subplots(figsize=(14, 6))
for bus in load_by_bus.columns:
    ax.plot(load_by_bus.index, -load_by_bus[bus], color=node_colors[bus], lw=0.8, alpha=0.75, label=f"load bus {bus}")

for carrier in gen_by_carrier.columns:
    lw = 1.4 if carrier != "load shedding" else 1.0
    ax.plot(gen_by_carrier.index, gen_by_carrier[carrier], color=tech_color(carrier), lw=lw, alpha=0.9, label=f"gen {carrier}")

ax.axhline(0, color="black", lw=0.8)
ax.set_title("Solved Load by Node and Generation by Technology")
ax.set_ylabel(f"MWh per model period ({period_label})")
ax.set_xlabel("Time")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small", ncol=1, frameon=True)
fig.autofmt_xdate(rotation=25)
plt.tight_layout()
plt.show()

annual_by_carrier = gen_by_carrier.sum().rename("generation_TWh") / 1e6
annual_load_by_bus = load_by_bus.sum().rename("load_TWh") / 1e6
print("Annual generation by carrier (TWh):")
display(styled_table(annual_by_carrier.to_frame(), "{:,.3f}"))
print("Annual load by bus (TWh):")
display(styled_table(annual_load_by_bus.to_frame(), "{:,.3f}"))


In [ ]:
# Nodal physical generation versus nodal load.
physical_generators = n.generators[n.generators.carrier != "load shedding"]
physical_dispatch_mwh = gen_dispatch_mwh.loc[:, physical_generators.index]
gen_by_bus = physical_dispatch_mwh.T.groupby(physical_generators.bus).sum().T
all_buses = sorted(set(load_by_bus.columns).union(gen_by_bus.columns), key=str)
node_palette = plt.cm.tab10(np.linspace(0, 1, max(len(all_buses), 1)))
node_colors = {bus: node_palette[i % len(node_palette)] for i, bus in enumerate(all_buses)}

fig, ax = plt.subplots(figsize=(14, 5.5))
for bus in all_buses:
    if bus in load_by_bus:
        ax.plot(load_by_bus.index, -load_by_bus[bus], color=node_colors[bus], lw=0.8, alpha=0.65, label=f"load bus {bus}")
    if bus in gen_by_bus:
        ax.plot(gen_by_bus.index, gen_by_bus[bus], color=node_colors[bus], lw=1.0, alpha=0.95, linestyle="--", label=f"gen bus {bus}")

ax.axhline(0, color="black", lw=0.8)
ax.set_title("Nodal Total Load and Physical Generation")
ax.set_ylabel(f"MWh per model period ({period_label})")
ax.set_xlabel("Time")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small", ncol=1, frameon=True)
fig.autofmt_xdate(rotation=25)
plt.tight_layout()
plt.show()

nodal_balance = pd.DataFrame({
    "annual_load_TWh": load_by_bus.sum() / 1e6,
    "annual_physical_generation_TWh": gen_by_bus.sum() / 1e6,
}).fillna(0.0)
nodal_balance["generation_minus_load_TWh"] = nodal_balance["annual_physical_generation_TWh"] - nodal_balance["annual_load_TWh"]
display(styled_table(nodal_balance, "{:,.3f}"))


## Wind and solar availability through the year

The weather files give an availability value between zero and one for each time and model location. The chart multiplies this value by the maximum capacity allowed at that location, then adds the locations together.

The dashed horizontal line is the total maximum capacity allowed by the land and sea assumptions. Faint lines show hourly values; the darker line is the daily average.

In [ ]:
profile_files = {
    "solar": PROFILE_DIR / "profile_solar.nc",
    "onwind": PROFILE_DIR / "profile_onwind.nc",
    "offwind-ac": PROFILE_DIR / "profile_offwind-ac.nc",
    "offwind-dc": PROFILE_DIR / "profile_offwind-dc.nc",
}

profile_rows = []
profile_series = {}
for tech, path in profile_files.items():
    if not path.exists():
        profile_rows.append({"technology": tech, "file": str(path.relative_to(REPO_ROOT)), "status": "missing"})
        continue
    ds = load_profile_dataset(path)
    if "profile" not in ds or "p_nom_max" not in ds:
        profile_rows.append({"technology": tech, "file": str(path.relative_to(REPO_ROOT)), "status": "no profile/p_nom_max"})
        continue
    p_nom_max = ds["p_nom_max"].fillna(0.0)
    total_p_nom_max = float(p_nom_max.sum())
    available = (ds["profile"].fillna(0.0) * p_nom_max).sum("bus").to_pandas()
    available.index = pd.to_datetime(available.index)
    profile_series[tech] = {"available_MW": available, "p_nom_max_MW": total_p_nom_max}
    mean_available = float(available.mean()) if len(available) else 0.0
    profile_rows.append({
        "technology": tech,
        "file": str(path.relative_to(REPO_ROOT)),
        "status": "ok",
        "hours": len(available),
        "profile_buses": int(ds.sizes.get("bus", 0)),
        "p_nom_max_MW": total_p_nom_max,
        "mean_available_MW": mean_available,
        "peak_available_MW": float(available.max()) if len(available) else 0.0,
        "mean_capacity_factor": mean_available / total_p_nom_max if total_p_nom_max > 0 else np.nan,
    })

profile_summary = pd.DataFrame(profile_rows).set_index("technology")
display(styled_table(profile_summary, {
    "p_nom_max_MW": "{:,.0f}",
    "mean_available_MW": "{:,.0f}",
    "peak_available_MW": "{:,.0f}",
    "mean_capacity_factor": "{:.2%}",
}))

positive = {tech: data for tech, data in profile_series.items() if data["p_nom_max_MW"] > 1e-6}
if not positive:
    print("No nonzero wind/solar profile capacities found.")
else:
    fig, axes = plt.subplots(len(positive), 1, figsize=(14, 3.2 * len(positive)), sharex=True)
    if len(positive) == 1:
        axes = [axes]
    for ax, (tech, data) in zip(axes, positive.items()):
        available = data["available_MW"]
        daily = available.resample("D").mean()
        max_cap = data["p_nom_max_MW"]
        ax.plot(available.index, available, color=tech_color(tech), lw=0.25, alpha=0.18, label="hourly available MW")
        ax.plot(daily.index, daily, color=tech_color(tech), lw=1.4, label="daily mean available MW")
        ax.axhline(max_cap, color="black", lw=0.9, linestyle="--", label=f"p_nom_max = {max_cap:,.0f} MW")
        ax.set_title(tech)
        ax.set_ylabel("MW")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper left", fontsize="small", frameon=True)
    axes[-1].set_xlabel("Time")
    fig.autofmt_xdate(rotation=25)
    plt.tight_layout()
    plt.show()


## Maps of maximum renewable capacity

Each renewable file contains:

- `potential[y, x]`: maximum capacity in each map cell after excluding unsuitable land or sea;
- `p_nom_max[bus]`: maximum capacity assigned to each model location. `bus` is PyPSA's term for a model connection point.

For this run, weather is represented on a 0.1-degree grid. Land and sea exclusions are first checked on a finer 100-metre grid and then added up to the weather grid and model regions.

In [ ]:
def _read_gdf(path):
    path = Path(path)
    if not path.exists():
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf.to_crs("EPSG:4326")


def _expanded_bounds(gdf, pad_frac=0.05):
    if gdf.empty:
        return None
    minx, miny, maxx, maxy = gdf.total_bounds
    dx = max(maxx - minx, 0.02)
    dy = max(maxy - miny, 0.02)
    return (minx - dx * pad_frac, maxx + dx * pad_frac, miny - dy * pad_frac, maxy + dy * pad_frac)


def _set_bounds(ax, gdf, pad_frac=0.05):
    bounds = _expanded_bounds(gdf, pad_frac)
    if bounds is None:
        return
    minx, maxx, miny, maxy = bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)


def _bus_points(network):
    buses = network.buses.copy()
    if "carrier" in buses:
        buses = buses[buses["carrier"].eq("AC")]
    buses = buses.dropna(subset=["x", "y"])
    return gpd.GeoDataFrame(
        buses,
        geometry=gpd.points_from_xy(buses["x"], buses["y"]),
        crs="EPSG:4326",
    )


country = _read_gdf(SHAPES_DIR / "country_shapes.geojson")
offshore = _read_gdf(SHAPES_DIR / "offshore_shapes.geojson")
onshore_regions = _read_gdf(BUS_REGION_DIR / "regions_onshore_elec_s_12flex.geojson")
offshore_regions = _read_gdf(BUS_REGION_DIR / "regions_offshore_elec_s_12flex.geojson")
for gdf in (onshore_regions, offshore_regions):
    if "name" in gdf:
        gdf["name"] = gdf["name"].astype(str)

bus_points = _bus_points(n)
profile_datasets = {}
for tech, path in profile_files.items():
    if not path.exists():
        continue
    ds = load_profile_dataset(path)
    if "potential" in ds:
        profile_datasets[tech] = ds

# Gridded heatmaps: MW of installable capacity in each weather grid cell.
techs = [tech for tech in profile_files if tech in profile_datasets]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
axes = axes.ravel()
for ax, tech in zip(axes, techs):
    ds = profile_datasets[tech]
    potential = ds["potential"].fillna(0.0)
    vmax = float(potential.quantile(0.98)) if potential.size else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    mesh = ax.pcolormesh(
        potential["x"].values,
        potential["y"].values,
        potential.values,
        shading="auto",
        cmap="viridis",
        vmin=0,
        vmax=vmax,
    )
    country.boundary.plot(ax=ax, color="black", linewidth=0.8)
    if tech.startswith("offwind") and not offshore.empty:
        offshore.boundary.plot(ax=ax, color="#4c78a8", linewidth=0.6, alpha=0.5)
        _set_bounds(ax, offshore, pad_frac=0.02)
    else:
        onshore_regions.boundary.plot(ax=ax, color="white", linewidth=0.5, alpha=0.8)
        _set_bounds(ax, country, pad_frac=0.08)
    if not bus_points.empty:
        bus_points.plot(ax=ax, color="black", markersize=10, zorder=5)
    total = float(ds["p_nom_max"].fillna(0.0).sum()) if "p_nom_max" in ds else 0.0
    ax.set_title(f"{tech}: gridded potential, total {total:,.0f} MW")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15)
    fig.colorbar(mesh, ax=ax, label="MW per cutout cell")
for ax in axes[len(techs):]:
    ax.axis("off")
plt.show()

# Nodal p_nom_max maps: maximum installable capacity assigned to each bus region.
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
axes = axes.ravel()
for ax, tech in zip(axes, techs):
    ds = profile_datasets[tech]
    regions = offshore_regions.copy() if tech.startswith("offwind") else onshore_regions.copy()
    pmax = ds["p_nom_max"].fillna(0.0).to_pandas().rename("p_nom_max_MW")
    regions = regions.merge(pmax, left_on="name", right_index=True, how="left")
    regions["p_nom_max_MW"] = regions["p_nom_max_MW"].fillna(0.0)
    vmax = regions["p_nom_max_MW"].max()
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    regions.plot(
        column="p_nom_max_MW",
        ax=ax,
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        edgecolor="#333333",
        linewidth=0.45,
        legend=True,
        legend_kwds={"label": "p_nom_max MW"},
    )
    country.boundary.plot(ax=ax, color="black", linewidth=0.8)
    if not bus_points.empty:
        bus_points.plot(ax=ax, color="black", markersize=10, zorder=5)
        for bus, row in bus_points.iterrows():
            ax.annotate(str(bus), (row.geometry.x, row.geometry.y), xytext=(3, 3), textcoords="offset points", fontsize=8)
    _set_bounds(ax, offshore if tech.startswith("offwind") and not offshore.empty else country, pad_frac=0.05)
    ax.set_title(f"{tech}: nodal p_nom_max")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15)
for ax in axes[len(techs):]:
    ax.axis("off")
plt.show()

## Land and sea assumptions for renewable energy

The table below reads the settings used for this run. With `potential: simple`, PyPSA-Earth multiplies the suitable area in each model region by the assumed MW per square kilometre. The result is the maximum wind or solar capacity allowed in that region.

In [ ]:
renewable_cfg = config.get("renewable", {})
assumption_rows = []
for tech in ["onwind", "solar", "offwind-ac", "offwind-dc"]:
    cfg = renewable_cfg.get(tech, {})
    copernicus = cfg.get("copernicus", {}) or {}
    ds = profile_datasets.get(tech)
    assumption_rows.append({
        "technology": tech,
        "cutout": cfg.get("cutout"),
        "resource_method": cfg.get("resource", {}).get("method"),
        "resource_detail": cfg.get("resource", {}).get("turbine") or cfg.get("resource", {}).get("panel"),
        "capacity_density_MW_per_km2": cfg.get("capacity_per_sqkm"),
        "copernicus_allowed_codes": ", ".join(map(str, copernicus.get("grid_codes", []))),
        "urban_buffer_m": copernicus.get("distance"),
        "natura_exclusion": cfg.get("natura"),
        "max_depth_m": cfg.get("max_depth"),
        "min_shore_distance_m": cfg.get("min_shore_distance"),
        "max_shore_distance_m": cfg.get("max_shore_distance"),
        "potential_method": cfg.get("potential"),
        "profile_clip_pu": cfg.get("clip_p_max_pu"),
        "profile_p_nom_max_MW": float(ds["p_nom_max"].fillna(0.0).sum()) if ds is not None and "p_nom_max" in ds else np.nan,
    })

renewable_assumptions = pd.DataFrame(assumption_rows).set_index("technology")
display(styled_table(renewable_assumptions, {
    "capacity_density_MW_per_km2": "{:,.2f}",
    "profile_clip_pu": "{:.3g}",
    "profile_p_nom_max_MW": "{:,.0f}",
}))

## OpenStreetMap transmission data used to create model locations

This chart shows the cleaned OpenStreetMap transmission features used to create the initial network, then overlays the smaller set of lines and locations used in the final result.

In [ ]:
def _network_line_gdf(network):
    rows = []
    buses = network.buses
    for line_id, row in network.lines.iterrows():
        geom = None
        raw_geom = row.get("geometry")
        if isinstance(raw_geom, str) and raw_geom.strip():
            try:
                geom = wkt.loads(raw_geom)
            except Exception:
                geom = None
        if geom is None:
            try:
                b0 = buses.loc[row["bus0"]]
                b1 = buses.loc[row["bus1"]]
                geom = gpd.GeoSeries.from_wkt([f"LINESTRING ({b0.x} {b0.y}, {b1.x} {b1.y})"], crs="EPSG:4326").iloc[0]
            except Exception:
                continue
        rows.append({
            "line": str(line_id),
            "bus0": row.get("bus0"),
            "bus1": row.get("bus1"),
            "s_nom_MW": row.get("s_nom", np.nan),
            "geometry": geom,
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


osm_lines = _read_gdf(BASE_NETWORK_DIR / "all_lines_build_network.geojson")
osm_buses = _read_gdf(BASE_NETWORK_DIR / "all_buses_build_network.geojson")
clustered_network = pypsa.Network(str(CLUSTERED_NETWORK_PATH)) if CLUSTERED_NETWORK_PATH.exists() else n
clustered_lines = _network_line_gdf(clustered_network)
clustered_buses = _bus_points(clustered_network)

osm_summary = pd.Series({
    "cleaned_osm_lines": len(osm_lines),
    "cleaned_osm_buses": len(osm_buses),
    "clustered_model_lines": len(clustered_lines),
    "clustered_model_ac_buses": len(clustered_buses),
}, name="count")
display(osm_summary.to_frame())

fig, ax = plt.subplots(figsize=(9, 9))
country.boundary.plot(ax=ax, color="black", linewidth=0.9, label="country shape")
if not osm_lines.empty:
    osm_lines.plot(ax=ax, color="#9aa0a6", linewidth=2.2, alpha=0.45, label="cleaned OSM lines")
if not osm_buses.empty:
    osm_buses.plot(ax=ax, color="#5f6368", markersize=28, alpha=0.7, label="cleaned OSM substations", zorder=4)
if not clustered_lines.empty:
    clustered_lines.plot(ax=ax, color="#d62728", linewidth=1.5, alpha=0.95, label="clustered PyPSA lines", zorder=5)
if not clustered_buses.empty:
    clustered_buses.plot(ax=ax, color="white", edgecolor="#d62728", linewidth=1.0, markersize=55, label="clustered PyPSA buses", zorder=6)
    for bus, row in clustered_buses.iterrows():
        ax.annotate(str(bus), (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords="offset points", fontsize=8, zorder=7)
_set_bounds(ax, country, pad_frac=0.08)
ax.set_title("Cleaned OSM transmission network and clustered PyPSA network")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend(loc="best", frameon=True)
plt.show()

## Input summary to share with project partners

In [ ]:
# Demand provenance from config and the run log.
load_options = config.get("load_options", {})
scenario = config.get("scenario", {})
atlite_cfg = config.get("atlite", {}).get("cutouts", {}).get("cutout-2013-era5", {})

gegis_path = "data/{ssp}/{prediction_year}/era5_{weather_year}/Africa.nc".format(
    ssp=load_options.get("ssp"),
    prediction_year=load_options.get("prediction_year"),
    weather_year=load_options.get("weather_year"),
)
if DEMAND_LOG.exists():
    log_text = DEMAND_LOG.read_text(errors="ignore")
    for line in log_text.splitlines():
        if "Merging demand data from paths" in line:
            gegis_path = line.split("paths", 1)[1].split("into", 1)[0].strip()
            break

input_rows = [
    {
        "input": "Demand source",
        "value": gegis_path,
        "notes": "GEGIS electricity demand, allocated to buses by GDP/population in build_demand_profiles.py",
    },
    {
        "input": "Demand output used by workflow",
        "value": str(DEMAND_PATH.relative_to(REPO_ROOT)) if DEMAND_PATH.exists() else "missing",
        "notes": "Hourly bus-level CSV before network simplification/clustering",
    },
    {
        "input": "Demand scenario",
        "value": f"{load_options.get('ssp')}, prediction_year={load_options.get('prediction_year')}, weather_year={load_options.get('weather_year')}, scale={load_options.get('scale')}",
        "notes": "From config load_options",
    },
    {
        "input": "Weather cutout",
        "value": "cutouts/cutout-2013-era5.nc",
        "notes": f"atlite module={atlite_cfg.get('module')}, dx={atlite_cfg.get('dx')}, dy={atlite_cfg.get('dy')}",
    },
    {
        "input": "Renewable profile files",
        "value": str(PROFILE_DIR.relative_to(REPO_ROOT)) if PROFILE_DIR.exists() else "missing",
        "notes": "Hourly NetCDF profiles with profile[time,bus] and p_nom_max[bus]",
    },
    {
        "input": "Solve temporal resolution",
        "value": ", ".join(scenario.get("opts", [])),
        "notes": f"Solved network has {len(n.snapshots)} snapshots and model period {period_label}",
    },
]
input_summary = pd.DataFrame(input_rows).set_index("input")
display(input_summary)

if DEMAND_PATH.exists():
    demand_raw = pd.read_csv(DEMAND_PATH, parse_dates=["time"])
    demand_cols = [c for c in demand_raw.columns if c != "time"]
    demand_stats = pd.Series({
        "file": str(DEMAND_PATH.relative_to(REPO_ROOT)),
        "rows_hours": len(demand_raw),
        "first_hour": str(demand_raw["time"].min()),
        "last_hour": str(demand_raw["time"].max()),
        "base_network_load_columns": len(demand_cols),
        "annual_demand_TWh_hourly_input": demand_raw[demand_cols].sum().sum() / 1e6,
        "peak_total_load_MW_hourly_input": demand_raw[demand_cols].sum(axis=1).max(),
        "annual_demand_TWh_solved_network": load_by_bus.sum().sum() / 1e6,
        "peak_total_load_MW_solved_network": n.loads_t.p_set.sum(axis=1).max(),
    })
    display(styled_table(demand_stats.to_frame("value"), "{}"))
else:
    print(f"Demand profile CSV not found: {DEMAND_PATH}")


In [ ]:
# Capacity limits used by the solved network.
gens = n.generators.copy()
gens["finite_p_nom_max"] = gens["p_nom_max"].replace(np.inf, np.nan)
gen_constraints = gens.groupby("carrier").agg(
    components=("carrier", "size"),
    p_nom_MW=("p_nom", "sum"),
    p_nom_min_MW=("p_nom_min", "sum"),
    p_nom_opt_MW=("p_nom_opt", "sum"),
    finite_p_nom_max_MW=("finite_p_nom_max", "sum"),
    unbounded_p_nom_max=("p_nom_max", lambda s: bool(np.isinf(s).any())),
).sort_index()
# If every row in a carrier is unbounded, the finite sum above is 0; show NaN instead.
for carrier, group in gens.groupby("carrier"):
    finite = group["finite_p_nom_max"].dropna()
    if finite.empty:
        gen_constraints.loc[carrier, "finite_p_nom_max_MW"] = np.nan

profile_pmax = profile_summary.get("p_nom_max_MW", pd.Series(dtype=float))
gen_constraints["profile_p_nom_max_MW"] = profile_pmax.reindex(gen_constraints.index)

display(styled_table(gen_constraints, {
    "p_nom_MW": "{:,.1f}",
    "p_nom_min_MW": "{:,.1f}",
    "p_nom_opt_MW": "{:,.1f}",
    "finite_p_nom_max_MW": "{:,.1f}",
    "profile_p_nom_max_MW": "{:,.1f}",
}))

other_rows = []
if not n.links.empty:
    links = n.links.copy()
    links["finite_p_nom_max"] = links["p_nom_max"].replace(np.inf, np.nan)
    for carrier, group in links.groupby("carrier"):
        finite = group["finite_p_nom_max"].dropna()
        other_rows.append({
            "component": "Link",
            "carrier": carrier,
            "components": len(group),
            "p_nom_opt_MW": group["p_nom_opt"].sum(),
            "finite_p_nom_max_MW": finite.sum() if not finite.empty else np.nan,
            "unbounded": bool(np.isinf(group["p_nom_max"]).any()),
        })
if not n.stores.empty:
    stores = n.stores.copy()
    stores["finite_e_nom_max"] = stores["e_nom_max"].replace(np.inf, np.nan)
    for carrier, group in stores.groupby("carrier"):
        finite = group["finite_e_nom_max"].dropna()
        other_rows.append({
            "component": "Store",
            "carrier": carrier,
            "components": len(group),
            "p_nom_opt_MW": np.nan,
            "finite_p_nom_max_MW": finite.sum() if not finite.empty else np.nan,
            "unbounded": bool(np.isinf(group["e_nom_max"]).any()),
        })

if other_rows:
    other_constraints = pd.DataFrame(other_rows).set_index(["component", "carrier"])
    display(styled_table(other_constraints, {
        "p_nom_opt_MW": "{:,.1f}",
        "finite_p_nom_max_MW": "{:,.1f}",
    }))
else:
    print("No link/store capacity constraints in this network.")

extendable = config.get("electricity", {}).get("extendable_carriers", {})
print("Configured extendable carriers:")
display(pd.Series({k: ", ".join(v) if isinstance(v, list) else v for k, v in extendable.items()}, name="carriers").to_frame())
